# Reaction-diffusion: training run analysis

Same architecture, training loop, and diagnostics as `experiments/lorenz63/analysis.ipynb`,
adapted to this experiment's differences from Lorenz:

- the field is genuinely 2D in space (`y = (y1, y2)`, `N = N_GRID**2 = 10_000` points) rather
  than a 1D Legendre-mode lift, so every plot below is a 2D heatmap/scatter rather than a line
  plot;
- there is no known low-dimensional ground-truth state to correlate against (`get_rd_data`
  never ships a `'z'` key -- see `training_data.py`), so the correlation-matrix /
  coordinate-recovery cells from the Lorenz notebook are replaced below by Champion et al.'s own
  relative-error metrics (`analyze_rd_model1.ipynb`, their `example_reactiondiffusion.py`) and by
  self-consistency checks (SINDy-simulated `z` vs. the encoder's own `z`) instead;
- `test_data` here is a single, temporally-ordered simulation trajectory (the strictly held-out
  last 10% of one long PDE integration -- temporal extrapolation, not interpolation), not a batch
  of independent (ic, timestep) snapshots like Lorenz's `validation_data`, so no reshaping into
  `(n_ics, n_steps, ...)` is needed anywhere below.

**Prerequisite**: `experiments/reaction_diffusion/reaction_diffusion.mat` must exist -- run
`rd_solver/reaction_diffusion.m` once in MATLAB first (see that script's header and
`training_data.py`'s docstring). The data-loading cell below raises `FileNotFoundError` until
it does.

In [1]:
import numpy as np
import jax
import jax.numpy as jnp
import optax
import pysindy as ps
import matplotlib.pyplot as plt
from scipy.integrate import odeint


In [2]:
import os
from pathlib import Path
# Move to the NeuralOperatorSINDy repo root regardless of where Jupyter started the
# kernel (this notebook lives at experiments/reaction_diffusion/analysis.ipynb).
if Path.cwd().name == "reaction_diffusion":
    os.chdir(Path.cwd().parents[1])
print(Path.cwd())


c:\Users\mra15\OneDrive\MSc2026\Code\NO_SINDy_NO\4 - NO-SINDy\NeuralOperatorSINDy


In [3]:
from src.training import (
    build_feature_library, theta_jax, train, save_checkpoint, load_checkpoint,
    sample_batch, sample_batch_subsampled, SINDyAE, PoolingEncoder, DeepSetPooling,
)
from experiments.reaction_diffusion.train import make_rd_decoder, RD_HIDDEN_WIDTH
from experiments.reaction_diffusion.config import Config
from experiments.reaction_diffusion.training_data import get_rd_data


## Data

In [4]:
cfg = Config()
rd_dir = Path.cwd() / "experiments" / "reaction_diffusion"

training_data, validation_data, test_data = get_rd_data(
    rd_dir / cfg.model.MAT_PATH,
    noise_strength=cfg.model.NOISE_STRENGTH,
    random_split=cfg.model.RANDOM_SPLIT,
    seed=cfg.model.SEED,
)
print(f"loaded {training_data['x'].shape[0]} train / {validation_data['x'].shape[0]} val / "
      f"{test_data['x'].shape[0]} test samples, N={training_data['x'].shape[1]} field points")
print(f"test trajectory spans t = {test_data['t'][0]:.2f} .. {test_data['t'][-1]:.2f}")


loaded 7999 train / 1000 val / 1000 test samples, N=10000 field points
test trajectory spans t = 450.05 .. 500.00


## Model

In [ ]:
lib = build_feature_library(cfg.model.POLY_ORDER, cfg.model.LATENT_DIM, cfg.model.INCLUDE_SINE)
N_FEATURES = lib.n_output_features_
FEAT_NAMES = list(lib.get_feature_names([f'z{i}' for i in range(cfg.model.LATENT_DIM)]))
print(f'Library: {N_FEATURES} features  {FEAT_NAMES}')

model = SINDyAE(
    encoder=PoolingEncoder(
        latent_dim=cfg.model.LATENT_DIM,
        is_variational=False,
        pooling_fn=DeepSetPooling(features=cfg.model.ENCODER_FEATURES),
    ),
    decoder=make_rd_decoder(cfg),
    n_features=N_FEATURES,
    latent_dim=cfg.model.LATENT_DIM,
    poly_order=cfg.model.POLY_ORDER,
    include_sine=cfg.model.INCLUDE_SINE,
)

## Train, or load a checkpoint\n\nRun **one** of the next two cells, not both.

In [6]:
key = jax.random.PRNGKey(cfg.model.SEED)
key, init_key, subsample_key = jax.random.split(key, 3)
rng = np.random.default_rng(cfg.model.SEED)

if cfg.model.SUBSAMPLE_POINTS:
    u0, udot0, x0 = sample_batch_subsampled(training_data, cfg.training.BATCH_SIZE, cfg.model.N_SUB, rng, subsample_key)
else:
    u0, udot0, x0 = sample_batch(training_data, cfg.training.BATCH_SIZE, rng)

mask = jnp.ones((N_FEATURES, cfg.model.LATENT_DIM))
params = model.init(init_key, u0, udot0, x0, mask)['params']

# Resume-aware: if a prior run (this cell, or experiments/reaction_diffusion/train.py)
# already checkpointed this config, this picks up from state.step -- a fast no-op if
# training already finished. Safe to re-run this cell any time, including after an SSH
# drop (run inside tmux/screen on the remote server so the underlying process survives a
# dropped connection regardless -- see checkpoint_every below as the safety net if it
# doesn't).
checkpoint_path = rd_dir / "checkpoint.pkl"
resume_from = load_checkpoint(checkpoint_path)
if resume_from is not None:
    print(f'found existing checkpoint at {checkpoint_path}, resuming')

state, mask, loss_hist = train(
    cfg, model, params, mask, training_data, rng,
    checkpoint_path=checkpoint_path,
    checkpoint_every=5_000,
    resume_from=resume_from,
    key=key,
)


JaxRuntimeError: RESOURCE_EXHAUSTED: Out of memory allocating 37427348164 bytes.

### Load a trained checkpoint instead of training

Run this cell **instead of** the training cell above (skip that one) if
`experiments/reaction_diffusion/checkpoint.pkl` already holds a finished (or in-progress) run --
e.g. copied over from the university GPU machine -- and you just want to analyze it without
resuming or continuing training. It reconstructs `state`/`mask`/`loss_hist` in the same shape the
training cell produces, so every cell below works unchanged either way. `loss_hist` is whatever
the checkpoint happened to have saved -- if it's empty or was stripped out (e.g. to keep the
transferred file small), the loss-curve cell below detects that and just skips the plot rather
than erroring.

In [ ]:
import types

checkpoint_path = rd_dir / "checkpoint.pkl"
ckpt = load_checkpoint(checkpoint_path)
if ckpt is None:
    raise FileNotFoundError(f"No checkpoint found at {checkpoint_path}")

state = types.SimpleNamespace(params=jax.tree_util.tree_map(jnp.asarray, ckpt['params']))
mask = jnp.asarray(ckpt['mask'])
loss_hist = ckpt.get('loss_hist', [])  # may be empty/excluded -- downstream cells handle that

total_steps = cfg.training.MAX_STEPS + cfg.training.REFINEMENT_STEPS
print(f"loaded checkpoint @ step {ckpt['step']}/{total_steps}")
print(f"active terms: {int(np.array(mask).sum())} / {mask.size}")
print(f"loss_hist entries available: {len(loss_hist)}")


## Results\n\n### Training loss

In [ ]:
if not loss_hist:
    print("no loss history available (checkpoint-only analysis) -- skipping loss curve plot")
else:
    loss_arr = {k: np.array([h[k] for h in loss_hist]) for k in loss_hist[0]}

    loss_names = ["loss", "loss_rec", "loss_dz", "loss_dx", "loss_sp", "loss_var"]
    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    for ax, name in zip(axes.flat, loss_names):
        ax.plot(loss_arr[name])
        ax.axvline(cfg.loss.THRESH_START, color="k", linestyle="--", alpha=0.4, label="thresholding starts")
        ax.set_yscale("log")
        ax.set_xlabel("step")
        ax.set_ylabel(name)
        ax.set_title(name)
        ax.legend()
    plt.suptitle("Training loss")
    plt.tight_layout()
    plt.show()


### Evaluation helper

Everything below re-uses one pass over `test_data` at its full (no-subsampling) grid resolution
-- `evaluate_full` chunks it through the model in `BATCH_SIZE`-sized pieces (the same chunking
pattern `sr3_refit` uses in `src/training.py`), since pushing the whole `(n_test, 10_000)` test
set through the encoder+decoder as one batch risks running out of memory.

In [ ]:
def evaluate_full(model, params, mask, data, batch_size):
    """
    Run the full (no-subsampling) forward pass over every row in `data`, in
    `batch_size`-sized chunks, concatenating every output back to host numpy arrays.
    """
    n = data['x'].shape[0]
    y = jnp.asarray(data['y_spatial'])
    outs = {k: [] for k in ['u_hat', 'z', 'dz_enc', 'dz_sindy', 'du_dec']}
    for start in range(0, n, batch_size):
        end = min(start + batch_size, n)
        u = jnp.asarray(data['x'][start:end])[..., None]
        u_dot = jnp.asarray(data['dx'][start:end])[..., None]
        x = jnp.broadcast_to(y[None], (end - start, *y.shape))
        u_hat, z, dz_enc, dz_sindy, du_dec, _ = model.apply({'params': params}, u, u_dot, x, mask)
        outs['u_hat'].append(np.array(u_hat))
        outs['z'].append(np.array(z))
        outs['dz_enc'].append(np.array(dz_enc))
        outs['dz_sindy'].append(np.array(dz_sindy))
        outs['du_dec'].append(np.array(du_dec))
    return {k: np.concatenate(v, axis=0) for k, v in outs.items()}


outs_test = evaluate_full(model, state.params, mask, test_data, cfg.training.BATCH_SIZE)
z_enc_traj = outs_test['z']  # (n_test, latent_dim) -- test_data is already one ordered trajectory

n_grid = int(np.sqrt(test_data['y_spatial'].shape[0]))
y_full = np.asarray(test_data['y_spatial'])
x_1d = y_full[:, 0].reshape(n_grid, n_grid)[0, :]
y_1d = y_full[:, 1].reshape(n_grid, n_grid)[:, 0]

def to_grid(u_flat):
    """Reshape a flat (N,) field back to (n_grid, n_grid), matching y_spatial's flatten order."""
    return np.asarray(u_flat).reshape(n_grid, n_grid)

print('evaluated test set:', {k: v.shape for k, v in outs_test.items()})


### Input field vs. reconstruction

A handful of test snapshots, spread across the held-out trajectory: true field, reconstruction,
and their absolute difference, each as a heatmap over the physical `(y1, y2)` domain.

In [ ]:
n_show = 4
show_idx = np.linspace(0, test_data['x'].shape[0] - 1, n_show).astype(int)

fig, axes = plt.subplots(3, n_show, figsize=(4 * n_show, 10.5))
for col, i in enumerate(show_idx):
    true_grid = to_grid(test_data['x'][i])
    rec_grid = to_grid(outs_test['u_hat'][i, :, 0])
    err_grid = np.abs(rec_grid - true_grid)

    im0 = axes[0, col].pcolormesh(x_1d, y_1d, true_grid, cmap='viridis', shading='auto')
    axes[0, col].set_title(f"true, t={test_data['t'][i]:.1f}")
    fig.colorbar(im0, ax=axes[0, col])

    im1 = axes[1, col].pcolormesh(x_1d, y_1d, rec_grid, cmap='viridis', shading='auto')
    axes[1, col].set_title('reconstruction')
    fig.colorbar(im1, ax=axes[1, col])

    im2 = axes[2, col].pcolormesh(x_1d, y_1d, err_grid, cmap='inferno', shading='auto')
    axes[2, col].set_title('|error|')
    fig.colorbar(im2, ax=axes[2, col])

    for row in range(3):
        axes[row, col].set_xlabel('y1')
        if col == 0:
            axes[row, col].set_ylabel('y2')

plt.suptitle('Input field vs. reconstruction (test set)')
plt.tight_layout()
plt.show()

print(f"reconstruction MSE over these {n_show} snapshots: "
      f"{np.mean((outs_test['u_hat'][show_idx, :, 0] - test_data['x'][show_idx]) ** 2):.3e}")


### Reconstruction error over the held-out trajectory

The snapshot grid above only shows `n_show` fixed instants. This tracks reconstruction MSE at
every timestep across the whole test trajectory, so any degradation over the forecast horizon
(test is always the strictly-held-out final 10% of the simulation) is visible directly.

In [ ]:
mse_t = np.mean((outs_test['u_hat'][:, :, 0] - test_data['x']) ** 2, axis=1)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(test_data['t'], mse_t)
ax.set_xlabel('t')
ax.set_ylabel('reconstruction MSE')
ax.set_title('Reconstruction error over the held-out test trajectory')
plt.tight_layout()
plt.show()

print(f"mean {mse_t.mean():.3e}   max {mse_t.max():.3e} at t={test_data['t'][mse_t.argmax()]:.2f}")


### Mesh-invariance: reconstruction from a subsampled point cloud

Exactly the check `training.py`'s `sample_batch_subsampled` exercises during training when
`cfg.model.SUBSAMPLE_POINTS=True`: encode a single test snapshot from only `cfg.model.N_SUB`
(out of the full `N_GRID**2`) randomly-chosen points, then query the decoder back out at the
*full* grid -- sparse in, dense out.

In [ ]:
if not cfg.model.SUBSAMPLE_POINTS:
    print('cfg.model.SUBSAMPLE_POINTS is False for this run -- nothing to show here.')
else:
    T_IDX = np.random.randint(test_data['x'].shape[0])
    u_full = test_data['x'][T_IDX]

    sub_key = jax.random.PRNGKey(np.random.randint(1_000_000))
    sub_idx = np.array(jax.random.choice(sub_key, y_full.shape[0], shape=(cfg.model.N_SUB,), replace=False))
    u_sub = jnp.asarray(u_full[sub_idx])[None, :, None]
    x_sub = jnp.asarray(y_full[sub_idx])[None, :, :]

    z_sub = model.encoder.apply({'params': state.params['encoder']}, u_sub, x_sub)
    x_full_query = jnp.asarray(y_full)[None, :, :]
    u_hat_from_sub = np.array(
        model.decoder.apply({'params': state.params['decoder']}, z_sub, x_full_query)
    )[0, :, 0]

    fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
    im0 = axes[0].pcolormesh(x_1d, y_1d, to_grid(u_full), cmap='viridis', shading='auto')
    axes[0].set_title(f"true field, t={test_data['t'][T_IDX]:.1f}")
    fig.colorbar(im0, ax=axes[0], label='u')

    axes[1].pcolormesh(x_1d, y_1d, to_grid(u_full), cmap='viridis', shading='auto', alpha=0.35)
    axes[1].scatter(y_full[sub_idx, 0], y_full[sub_idx, 1], c='red', s=4)
    axes[1].set_title(f'subsampled input ({cfg.model.N_SUB}/{y_full.shape[0]} pts, red)')

    im2 = axes[2].pcolormesh(x_1d, y_1d, to_grid(u_hat_from_sub), cmap='viridis', shading='auto')
    axes[2].set_title('reconstruction at full resolution')
    fig.colorbar(im2, ax=axes[2], label='u')

    for ax in axes:
        ax.set_xlabel('y1'); ax.set_ylabel('y2')
    plt.tight_layout()
    plt.show()

    mse_sub = np.mean((u_hat_from_sub - u_full) ** 2)
    print(f'reconstruction MSE (full-res output from {cfg.model.N_SUB}-pt sparse input): {mse_sub:.3e}')


## Latent space

`z` here is not benchmarked against any known ground-truth low-dimensional state (unlike
Lorenz's synthetic lift -- reaction-diffusion has no such thing; see `get_rd_data`'s docstring),
so this only looks at the encoded trajectory itself: its time series and its phase portrait,
matching Champion et al.'s own `analyze_rd_model1.ipynb` (their cells 6-7).

In [ ]:
n_panels = cfg.model.LATENT_DIM + (1 if cfg.model.LATENT_DIM >= 2 else 0)
fig, axes = plt.subplots(1, n_panels, figsize=(5 * n_panels, 4))
axes = np.atleast_1d(axes)
for d in range(cfg.model.LATENT_DIM):
    axes[d].plot(test_data['t'], z_enc_traj[:, d])
    axes[d].set_xlabel('t')
    axes[d].set_ylabel(f'z[{d}]')
    axes[d].set_title(f'encoded z[{d}](t)')

if cfg.model.LATENT_DIM >= 2:
    axes[-1].plot(z_enc_traj[:, 0], z_enc_traj[:, 1], linewidth=1.5)
    axes[-1].set_xlabel('z[0]')
    axes[-1].set_ylabel('z[1]')
    axes[-1].set_title('phase portrait (z[0] vs z[1])')
    axes[-1].axis('equal')

plt.tight_layout()
plt.show()


## SINDy consistency: relative errors

Champion et al.'s own metrics (`analyze_rd_model1.ipynb`, their final cell), reproduced here
directly instead of the raw (non-normalized) loss terms in the loss curves above -- each is the
corresponding squared error normalized by the mean square of the quantity being predicted, so it
reads as a scale-free fraction rather than depending on the field's raw units:

- `decoder_x_error`: reconstruction, `u_hat` vs. `u`
- `decoder_dx_error`: decoder-propagated SINDy derivative, `du_dec` vs. the true `du/dt`
- `sindy_dz_error`: encoder-Jacobian derivative vs. the SINDy-predicted derivative, `dz_enc` vs.
  `dz_sindy`

In [ ]:
decoder_x_error = np.mean((test_data['x'] - outs_test['u_hat'][:, :, 0]) ** 2) / np.mean(test_data['x'] ** 2)
decoder_dx_error = np.mean((test_data['dx'] - outs_test['du_dec'][:, :, 0]) ** 2) / np.mean(test_data['dx'] ** 2)
sindy_dz_error = np.mean((outs_test['dz_enc'] - outs_test['dz_sindy']) ** 2) / np.mean(outs_test['dz_enc'] ** 2)

print('Decoder relative error: %f' % decoder_x_error)
print('Decoder relative SINDy error: %f' % decoder_dx_error)
print('SINDy relative error, z: %f' % sindy_dz_error)


## Discovered equations

In [ ]:
def format_equations(xi, feat_names, latent_dim, threshold=1e-3, precision=3):
    lines = []
    for d in range(latent_dim):
        terms = [
            f"{coef:+.{precision}f} {name}"
            for coef, name in zip(xi[:, d], feat_names)
            if abs(coef) >= threshold
        ]
        rhs = " ".join(terms) if terms else "0"
        lines.append(f"dz{d}/dt = {rhs}")
    return "\n".join(lines)


xi_final = np.array(state.params['xi']) * np.array(mask)
print(format_equations(xi_final, FEAT_NAMES, cfg.model.LATENT_DIM))
print()
print(f"active terms: {int(np.array(mask).sum())} / {mask.size}")


## Forecasting: full trajectory

Integrate the discovered SINDy ODE (`theta(z) @ xi`, using the trained + thresholded mask)
forward from the encoder's own `z[0]` across the *entire* test trajectory, and compare against
the encoder's own `z(t)` -- the same comparison as Champion et al.'s `analyze_rd_model1.ipynb`
cells 5-7 (`sindy_simulate` there, `odeint` here).

In [ ]:
def sindy_rhs(z, t):
    theta = np.array(theta_jax(
        jnp.asarray(z)[None, :], cfg.model.POLY_ORDER, cfg.model.LATENT_DIM, cfg.model.INCLUDE_SINE
    ))[0]
    return theta @ xi_final


z_sim = odeint(sindy_rhs, z_enc_traj[0], test_data['t'])

fig, axes = plt.subplots(cfg.model.LATENT_DIM, 1, figsize=(7, 2.5 * cfg.model.LATENT_DIM), sharex=True)
axes = np.atleast_1d(axes)
for d in range(cfg.model.LATENT_DIM):
    axes[d].plot(test_data['t'], z_enc_traj[:, d], color='#888888', linewidth=2, label='encoder')
    axes[d].plot(test_data['t'], z_sim[:, d], '--', linewidth=2, label='SINDy simulation')
    axes[d].set_ylabel(f'z[{d}]')
axes[0].legend()
axes[-1].set_xlabel('t')
plt.suptitle('Full-trajectory SINDy simulation vs. encoder')
plt.tight_layout()
plt.show()

if cfg.model.LATENT_DIM >= 2:
    fig, ax = plt.subplots(figsize=(4, 4))
    ax.plot(z_sim[:, 0], z_sim[:, 1], linewidth=2)
    ax.set_title('SINDy-simulated phase portrait')
    ax.axis('equal')
    plt.tight_layout()
    plt.show()

print(f"||z_sim - z_enc|| / ||z_enc|| over the full trajectory: "
      f"{np.linalg.norm(z_sim - z_enc_traj) / np.linalg.norm(z_enc_traj):.3e}")


## Forecasting: held-out half

A stricter test than the full-trajectory simulation above: integrate only from the trajectory's
midpoint, using nothing after it, and check whether the SINDy model still tracks the encoder's
own trajectory through the second half it was never given.

In [ ]:
T_OBS = len(test_data['t']) // 2

z0_fc = z_enc_traj[T_OBS - 1]
t_forecast = test_data['t'][T_OBS:] - test_data['t'][T_OBS - 1]
z_forecast = odeint(sindy_rhs, z0_fc, t_forecast)

fig, axes = plt.subplots(1, cfg.model.LATENT_DIM, figsize=(5 * cfg.model.LATENT_DIM, 4))
axes = np.atleast_1d(axes)
for d, ax in enumerate(axes):
    ax.plot(test_data['t'][:T_OBS], z_enc_traj[:T_OBS, d], color='#888888', label='encoder (observed)')
    ax.plot(test_data['t'][T_OBS:], z_enc_traj[T_OBS:, d], color='tab:blue', label='encoder (held out)')
    ax.plot(test_data['t'][T_OBS:], z_forecast[:, d], '--', color='tab:red', label='SINDy forecast')
    ax.set_title(f'z[{d}]')
    ax.set_xlabel('t')
    ax.legend()
plt.tight_layout()
plt.show()

fc_err = np.linalg.norm(z_forecast - z_enc_traj[T_OBS:]) / np.linalg.norm(z_enc_traj[T_OBS:])
print(f"||z_forecast - z_enc|| / ||z_enc|| over the held-out half: {fc_err:.3e}")


## Mesh-density consistency

Reaction-diffusion's field is fixed at the solver's native `N_GRID x N_GRID` resolution --
unlike Lorenz's analytically-defined lift, there is no way to query the *true* field at a finer
resolution than the PDE was solved at, so this can't repeat Lorenz's supersampling convergence
check (`experiments/lorenz63/analysis.ipynb`'s final cell). What it can test instead: does the
encoder give (approximately) the same `z` for the same underlying field regardless of how many
of its `N_GRID**2` points it's given to look at? Several random subsample sizes, a handful of
held-out snapshots each, encoded and compared against encoding the same snapshots at the full
grid.

In [ ]:
N_SUB_LIST = sorted(set(int(f * y_full.shape[0]) for f in [0.01, 0.03, 0.1, 0.3, 1.0]))
N_SUB_LIST = [n for n in N_SUB_LIST if n >= 1]
N_SNAPSHOTS = min(20, test_data['x'].shape[0])

snap_idx = np.random.default_rng(0).choice(test_data['x'].shape[0], size=N_SNAPSHOTS, replace=False)
u_snap = jnp.asarray(test_data['x'][snap_idx])

def encode_at_n_sub(n_sub, key):
    keys = jax.random.split(key, N_SNAPSHOTS)

    def pick(k, u_row):
        idx = jax.random.choice(k, y_full.shape[0], shape=(n_sub,), replace=False)
        return u_row[idx], jnp.asarray(y_full)[idx]

    u_s, x_s = jax.vmap(pick)(keys, u_snap)
    z_s = model.encoder.apply({'params': state.params['encoder']}, u_s[..., None], x_s)
    return np.array(z_s)


z_ref = encode_at_n_sub(y_full.shape[0], jax.random.PRNGKey(0))  # full grid, no subsampling

means, stds = [], []
for n_sub in N_SUB_LIST:
    z_n = encode_at_n_sub(n_sub, jax.random.PRNGKey(n_sub))
    rel = np.linalg.norm(z_n - z_ref, axis=1) / (np.linalg.norm(z_ref, axis=1) + 1e-12)
    means.append(rel.mean())
    stds.append(rel.std())
    print(f'n_sub={n_sub:6d}   mean rel. disagreement = {rel.mean():.4e}   (std {rel.std():.4e})')

means, stds = np.array(means), np.array(stds)

fig, ax = plt.subplots(figsize=(7, 5))
ax.fill_between(N_SUB_LIST, means - stds, means + stds, alpha=0.2)
ax.plot(N_SUB_LIST, means, 'o-')
ax.axvline(cfg.model.N_SUB, color='0.5', linestyle='--', label='training N_SUB')
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('n_sub (points encoder sees)')
ax.set_ylabel(r'$\|z^{(n)} - z^{(full)}\| / \|z^{(full)}\|$')
ax.set_title('Encoder mesh-density consistency')
ax.legend()
plt.tight_layout()
plt.show()
